In [1]:
import requests
import time
import pandas
from concurrent.futures import ThreadPoolExecutor, as_completed
url = "http://ws.audioscrobbler.com/2.0/"
API_KEY = "bd5cff0c8431c3b8888f8db83ed41ace"



df = pandas.read_csv(r'C:\Users\ryanw\CSCI413\artist_names.csv')

In [2]:
def getTopTracks(artistName):
    params = {
        "method": "artist.gettoptracks",
        "artist": artistName,
        "api_key": API_KEY,
        "format": "json",
        "limit": 10
    }
    response = requests.get(url, params=params).json()
    tracks = response.get("toptracks", {}).get("track", [])
    results = []
    for track in tracks:
        
        results.append({
            "artist": artistName,
            "track": track["name"],
            "listeners": track.get("listeners"),
            "playcount": track.get("playcount")
        })
    # for track in tracks:
    #     track_name = track["name"]
    #     listeners = track.get("listeners")
    #     playcount = track.get("playcount")

    #     # 3. Get top tags for the track
    #     tags = []
    #     track_info_params = {
    #         "method": "track.getInfo",
    #         "artist": artistName,
    #         "track": track_name,
    #         "api_key": API_KEY,
    #         "format": "json"
    #     }
    #     track_info = requests.get(url, params=track_info_params).json()
    #     if "track" in track_info and "toptags" in track_info["track"]:
    #         tags = [tag["name"] for tag in track_info["track"]["toptags"].get("tag", [])]

    #     results.append({
    #         "artist": artistName,
    #         "track": track_name,
    #         "listeners": listeners,
    #         "playcount": playcount,
    #         "tags": ", ".join(tags)
    #     })
    time.sleep(.25)
    return results

In [3]:
all_results = []
with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(getTopTracks, artist): artist for artist in df['artist_name']}

        for future in as_completed(futures):
            artist = futures[future]
            try:
                results = future.result()
                all_results.extend(results)
                print(f"Finished {artist} ({len(results)} tracks)")
            except Exception as e:
                print(f"Error with {artist}: {e}")

Finished The Weeknd (10 tracks)
Finished Taylor Swift (10 tracks)
Finished Radiohead (10 tracks)
Finished Kendrick Lamar (10 tracks)
Finished PinkPantheress (10 tracks)
Finished Kanye West (10 tracks)
Finished Rihanna (10 tracks)
Finished Charli xcx (10 tracks)
Finished Drake (10 tracks)
Finished Lady Gaga (10 tracks)
Finished Tame Impala (10 tracks)
Finished Tyler, The Creator (10 tracks)
Finished Ariana Grande (10 tracks)
Finished Lana Del Rey (10 tracks)
Finished Sabrina Carpenter (10 tracks)
Finished Bad Bunny (10 tracks)
Finished Bruno Mars (10 tracks)
Finished Arctic Monkeys (10 tracks)
Finished Gorillaz (10 tracks)
Finished Billie Eilish (10 tracks)
Finished Frank Ocean (10 tracks)
Finished Harry Styles (10 tracks)
Finished Mitski (10 tracks)
Finished Paramore (10 tracks)
Finished A$AP Rocky (10 tracks)
Finished Doja Cat (10 tracks)
Finished Don Toliver (10 tracks)
Finished Chappell Roan (10 tracks)
Finished Fleetwood Mac (10 tracks)
Finished The Smiths (10 tracks)
Finished Link

In [9]:
songs = pandas.DataFrame(all_results)
songs.to_csv('songs.csv', index = False)